# Finance Risk Analytics: Data Extraction, Validation & Reporting
**Author:** Boyinapalli Phani Shankar | **Dataset:** 7,043 records

**Pipeline:** Data Extraction → Transformation → Validation → SQL Analysis → Modelling → Reporting


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
sns.set_style('whitegrid')
print('Libraries loaded.')

Libraries loaded.


## 1. Data Extraction & Transformation


In [ ]:
# Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# Place at: data/Telco-Customer-Churn.csv
df = pd.read_csv('data/Telco-Customer-Churn.csv')
print(f'Records: {len(df):,} | Columns: {df.shape[1]}')
df.head(3)

Records: 7,043 | Columns: 21


In [ ]:
# Transformation: coerce types, impute, feature engineering
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing = df['TotalCharges'].isnull().sum()
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.drop(columns=['customerID'], inplace=True)

df['tenure_band'] = pd.cut(df['tenure'], bins=[-1,3,12,24,48,999],
    labels=['0-3 mo','4-12 mo','13-24 mo','25-48 mo','49+ mo'])
df['charge_band'] = pd.cut(df['MonthlyCharges'], bins=[0,35,65,95,999],
    labels=['Low','Medium','High','Very High'])
service_cols = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
    'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['service_count'] = df[service_cols].apply(
    lambda r: sum(v not in ['No','No internet service','No phone service'] for v in r), axis=1)
df['contract_ordinal'] = df['Contract'].map({'Month-to-month':0,'One year':1,'Two year':2})
print(f'Imputed missing TotalCharges: {missing} | Shape after transform: {df.shape}')

Imputed missing TotalCharges: 11 | Shape after transform: (7043, 24)


## 2. Validation & Data Quality


In [ ]:
RULES = {'MonthlyCharges':(0,200), 'TotalCharges':(0,15000), 'tenure':(0,120)}
df['anomaly_flag'] = False
for col,(lo,hi) in RULES.items():
    mask = (df[col]<lo)|(df[col]>hi)
    df.loc[mask,'anomaly_flag'] = True

report = {'total_records':len(df), 'null_count':int(df.isnull().sum().sum()),
    'anomaly_count':int(df['anomaly_flag'].sum()),
    'completeness_pct':round(100*(1-df.isnull().sum().sum()/df.size),2)}
print('Data Quality Report'); print('-'*32)
for k,v in report.items(): print(f'  {k}: {v}')

Data Quality Report
--------------------------------
  total_records: 7043
  null_count: 0
  anomaly_count: 0
  completeness_pct: 100.0


## 3. SQL Cohort Analysis
Pandas equivalent of the 11 SQL queries in `sql/queries.sql`


In [ ]:
total   = len(df)
churned = (df['Churn']=='Yes').sum()
print(f'Total records: {total:,} | Churned: {churned:,} ({churned/total*100:.1f}%)')
print()

# Q3: Churn by contract
q3 = df.groupby('Contract')['Churn'].apply(
    lambda x: round(100*(x=='Yes').sum()/len(x),1)).reset_index()
q3.columns=['Contract','Churn Rate (%)']
print('Churn Rate by Contract:')
print(q3.sort_values('Churn Rate (%)',ascending=False).to_string(index=False))
print()

# Q2: High-risk cohort
hr = df[(df['Contract']=='Month-to-month')&(df['tenure']<=3)&(df['Churn']=='Yes')]
print(f'High-risk cohort: {len(hr):,} = {len(hr)/churned*100:.1f}% of all churned')

Total records: 7,043 | Churned: 1,869 (26.5%)

Churn Rate by Contract:
        Contract  Churn Rate (%)
Month-to-month            42.7
      One year            11.3
      Two year             2.8

High-risk cohort: 716 = 38.3% of all churned


## 4. Exploratory Data Analysis


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))

# Churn distribution
churn_c = df['Churn'].value_counts()
axes[0].bar(churn_c.index, churn_c.values, color=['#2ecc71','#e74c3c'])
axes[0].set_title('Churn Distribution',fontweight='bold')
for i,v in enumerate(churn_c.values): axes[0].text(i,v+30,f'{v:,}\n({v/total*100:.1f}%)',ha='center')

# Churn by contract
cc = df.groupby('Contract')['Churn'].apply(lambda x:(x=='Yes').sum()/len(x)*100)
axes[1].bar(cc.index,cc.values,color=['#e74c3c','#f39c12','#2ecc71'])
axes[1].set_title('Churn Rate by Contract',fontweight='bold'); axes[1].set_ylabel('Churn Rate (%)')

# Monthly charges distribution
for label,color in [('No','#2ecc71'),('Yes','#e74c3c')]:
    axes[2].hist(df[df['Churn']==label]['MonthlyCharges'],bins=30,alpha=0.5,color=color,label=f'Churn={label}',density=True)
axes[2].set_title('Monthly Charges by Churn',fontweight='bold'); axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/figures/01_eda_overview.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: outputs/figures/01_eda_overview.png')

## 5. Modelling: LR vs Random Forest vs XGBoost


In [ ]:
df_m = df.drop(columns=['tenure_band','charge_band','anomaly_flag'],errors='ignore').copy()
for col in df_m.select_dtypes(include='object').columns:
    if col!='Churn': df_m[col] = LabelEncoder().fit_transform(df_m[col].astype(str))
df_m['Churn'] = (df_m['Churn']=='Yes').astype(int)
X=df_m.drop(columns=['Churn']); y=df_m['Churn']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200,random_state=42),
    'XGBoost':              xgb.XGBClassifier(n_estimators=200,max_depth=5,
                                use_label_encoder=False,eval_metric='logloss',random_state=42),
}
results={}
print(f'{"Model":<25} {"AUC":>7} {"F1":>7} {"Acc":>7}'); print('-'*50)
for name,model in models.items():
    model.fit(X_tr,y_tr)
    proba=model.predict_proba(X_te)[:,1]; preds=model.predict(X_te)
    r={'auc':round(roc_auc_score(y_te,proba),3),'f1':round(f1_score(y_te,preds,average='weighted'),3),
       'acc':round(accuracy_score(y_te,preds),3),'proba':proba,'model':model}
    results[name]=r
    print(f'{name:<25} {r["auc"]:>7} {r["f1"]:>7} {r["acc"]:>7}')

Model                       AUC      F1     Acc
--------------------------------------------------
Logistic Regression       0.843   0.791   0.783
Random Forest             0.851   0.800   0.800
XGBoost                   0.870   0.813   0.810


## 6. Business Reporting & High-Risk Flagging


In [ ]:
df_m['churn_proba'] = results['XGBoost']['model'].predict_proba(X)[:,1]
df_m['high_risk']   = df_m['churn_proba']>=0.5
hr_n = df_m['high_risk'].sum()
print(f'High-risk records flagged: {hr_n:,} ({hr_n/len(df_m)*100:.1f}% of total)')
print()
print('Business Recommendations (for regulatory reporting):')
print('  1. Early-tenure intervention - 0-3 month month-to-month customers = 38% of churn')
print('  2. Contract upgrade campaign  - offer 1-year contract at month 2-3')
print('  3. Payment method migration   - electronic-check has highest churn rate')
print('  4. Fiber optic bundle         - TechSupport + OnlineSecurity lowers churn probability')
print()
print('Pipeline complete. Validation rules passed. Output ready for Power BI dashboard.')

High-risk records flagged: 2,680 (38.1% of total)

Business Recommendations (for regulatory reporting):
  1. Early-tenure intervention - 0-3 month month-to-month customers = 38% of churn
  2. Contract upgrade campaign  - offer 1-year contract at month 2-3
  3. Payment method migration   - electronic-check has highest churn rate
  4. Fiber optic bundle         - TechSupport + OnlineSecurity lowers churn probability

Pipeline complete. Validation rules passed. Output ready for Power BI dashboard.
